# CG Prediction: deploy the trained model on unknown flights

This notebook loads the model trained in `nn.Linear_test.ipynb` and uses it to guess the CG of flights it has never seen.

What it does:
1. Converts raw flight-controller `.xlsx` files to CSV, the same way training did (`folder_cleaner`, `downsample=True`)
2. Normalizes them with the **training** scaler (never refit on unknown data)
3. Cuts each flight into overlapping windows, predicts every window, then combines them into one answer per flight
4. Shows how confident each answer is, and flags flights that look different from the training data
5. If a filename contains a CG code (like `0.45A`), it scores the guess against it. Unknown files can have any name.

**Rule:** the settings in the CONFIG cell must match the ones used to train the model.

In [ ]:
#Mitchell Moffitt
#Deploy / inference notebook for the 1D CG model

import os, glob, re, io, contextlib, pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import fennec_ml as fn
from sklearn.preprocessing import StandardScaler

from Variable_CG_Preprocessing import (Flight_Objects_Var, split_flights_by_fraction,
                                       extract_cg_label, unscale_target, CG_MAX, CG_STEP)

### Config
Point `unknown_dir` at a folder of raw `.xlsx` flight logs.

In [ ]:
# --- CONFIG ---
goal3_dir = r"C:\Users\Mitch\OneDrive - LeTourneau University\Andrew Davis's files - Project FENNEC\FENNEC 2026-2027\Data Science Team\Mitchell Moffitt\Python Code\Jupyter Notebooks\GOAL 3- Data preprocessing"

unknown_dir = os.path.join(goal3_dir, "Unknown_flights")          # raw .xlsx files to predict
unknown_csv_dir = os.path.join(unknown_dir, "processed")           # CSVs get written here
results_path = os.path.join(unknown_dir, "CG_predictions.csv")     # table of answers

model_path = os.path.join("saved_models", "Best_1DVar_Model.pth")
scaler_path = os.path.join("saved_models", "scaler.pkl")
vars_json = "1D_Variable_vars_of_interest.json"

# Only used to rebuild the scaler if scaler.pkl doesn't exist yet.
# MUST match the training notebook exactly (same folder, fractions and seed),
# otherwise the scaler is fit on different flights than the model was.
train_csv_dir = os.path.join(goal3_dir, "Processed_data_realistic")
TRAIN_FRAC, VAL_FRAC, SPLIT_SEED = 0.7, 0.15, 42

# Model layer sizes are read from the saved file; these must match the training notebook
DROPOUT_RATE1, DROPOUT_RATE2 = 0.1, 0.3

PREDICT_STRIDE_DIV = 4      # stride = segment_length // 4 -> overlapping windows, more votes per flight

### Load the model
The layer sizes come straight from the saved weights, so they can't get out of sync.

In [ ]:
class LinearRegressionModelV0(nn.Module):   # same architecture as the training notebook
    def __init__(self, INPUT_FEATURES, HIDDEN_LAYER1, HIDDEN_LAYER2, OUTPUT_FEATURES, DROPOUT_RATE1, DROPOUT_RATE2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(INPUT_FEATURES, HIDDEN_LAYER1), nn.ReLU(), nn.Dropout(DROPOUT_RATE1),
            nn.Linear(HIDDEN_LAYER1, HIDDEN_LAYER2),  nn.ReLU(), nn.Dropout(DROPOUT_RATE2),
            nn.Linear(HIDDEN_LAYER2, OUTPUT_FEATURES))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.flatten(start_dim=1)
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
state = torch.load(model_path, map_location=device)

HIDDEN_LAYER1, INPUT_FEATURES = state["net.0.weight"].shape
HIDDEN_LAYER2 = state["net.3.weight"].shape[0]

with open(vars_json) as f:
    import json
    vars_of_interest = json.load(f)
INPUT_CHANNELS = sum(len(c) for c in vars_of_interest.values())

if INPUT_FEATURES % INPUT_CHANNELS:
    raise ValueError(f"Model expects {INPUT_FEATURES} inputs, which isn't a multiple of {INPUT_CHANNELS} channels. "
                     "The vars-of-interest JSON doesn't match the one used in training.")
SEGMENT_LENGTH = INPUT_FEATURES // INPUT_CHANNELS

model = LinearRegressionModelV0(INPUT_FEATURES, HIDDEN_LAYER1, HIDDEN_LAYER2, 1, DROPOUT_RATE1, DROPOUT_RATE2).to(device)
model.load_state_dict(state)
model.eval()                                  # dropout off - always for predictions

print(f"Model loaded: {INPUT_CHANNELS} channels x {SEGMENT_LENGTH} rows per window "
      f"({SEGMENT_LENGTH / 10:.0f} s at 10 Hz), hidden layers {HIDDEN_LAYER1}/{HIDDEN_LAYER2}")

### Load the training scaler
The model only understands data scaled exactly like its training data. The training notebook saves `scaler.pkl` next to the model every time it trains, so this just loads it. If the file is missing (a model trained before that was added), it's rebuilt from the same training flights.

In [ ]:
if os.path.exists(scaler_path):
    with open(scaler_path, "rb") as f:
        scaler = pickle.load(f)
    print(f"Loaded scaler from {scaler_path}")
else:
    train_flights_all = Flight_Objects_Var(train_csv_dir)
    train_flights, _, _ = split_flights_by_fraction(train_flights_all, train_frac=TRAIN_FRAC,
                                                    val_frac=VAL_FRAC, seed=SPLIT_SEED)
    scaler = StandardScaler().fit(np.vstack([f.data for f in train_flights]))
    with open(scaler_path, "wb") as f:
        pickle.dump(scaler, f)
    print(f"Rebuilt scaler from {len(train_flights)} training flights and saved it to {scaler_path}")

if scaler.n_features_in_ != INPUT_CHANNELS:
    raise ValueError(f"Scaler has {scaler.n_features_in_} features but the model uses {INPUT_CHANNELS}")

### Convert the unknown flights to CSV
Same settings as training. Files that fail (missing sheets or columns) are skipped and listed.

In [ ]:
os.makedirs(unknown_csv_dir, exist_ok=True)
xlsx_files = sorted(glob.glob(os.path.join(unknown_dir, "*.xlsx")))
if not xlsx_files:
    raise FileNotFoundError(f"No .xlsx files in {unknown_dir}")

failed = []
for path in xlsx_files:
    csv_path = os.path.join(unknown_csv_dir, os.path.basename(path)[:-5] + ".csv")
    if os.path.exists(csv_path) and os.path.getmtime(csv_path) > os.path.getmtime(path):
        continue                                   # already converted and up to date
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            fn.data_cleaner(path, unknown_csv_dir, overwrite=True, varspath=vars_json, downsample=True)
    except Exception as e:
        failed.append((os.path.basename(path), str(e)))

print(f"{len(xlsx_files) - len(failed)} of {len(xlsx_files)} flights ready")
for name, err in failed:
    print(f"  SKIPPED {name}: {err[:150]}")

### Predict
Each flight is cut into overlapping windows and every window gets a prediction. The flight's answer is the **median** of its windows, which ignores a few bad windows (like time on the ground).

- **Agreement:** the share of windows that snap to the same CG position as the final answer. Treat a low agreement as a guess, not an answer.
- **Data check:** compares each flight's channels to the training data. A big difference usually means a different helicopter, different units, or a channel in the wrong order.

In [ ]:
CG_LEVELS = np.round(np.arange(-CG_MAX, CG_MAX + 1e-9, CG_STEP), 2)
snap = lambda v: np.clip(np.round(np.asarray(v) / CG_STEP) * CG_STEP, -CG_MAX, CG_MAX)

def cg_name(x):
    """+0.45 -> '0.45F', -0.30 -> '0.30A', 0 -> '0.00M'."""
    x = round(float(x), 2)
    return "0.00M" if x == 0 else f"{abs(x):.2f}{'F' if x > 0 else 'A'}"

def true_label(name):
    """CG from the filename if it has one (e.g. 342B_0.45A_H), else None."""
    try:
        return extract_cg_label(name + ".csv")   # needs an extension, or "0.45F_L" looks like one
    except ValueError:
        return None

stride = max(1, SEGMENT_LENGTH // PREDICT_STRIDE_DIV)
flights = []                                   # one dict per flight, used by every plot below
for csv_path in sorted(glob.glob(os.path.join(unknown_csv_dir, "*.csv"))):
    name = os.path.basename(csv_path)[:-4]
    raw = pd.read_csv(csv_path).values
    if raw.shape[1] != INPUT_CHANNELS:
        print(f"SKIPPED {name}: has {raw.shape[1]} columns, model needs {INPUT_CHANNELS}")
        continue
    if raw.shape[0] < SEGMENT_LENGTH:
        print(f"SKIPPED {name}: only {raw.shape[0]} rows, a window needs {SEGMENT_LENGTH}")
        continue

    data = scaler.transform(raw).astype(np.float32)
    starts = np.arange(0, len(data) - SEGMENT_LENGTH + 1, stride)
    windows = np.stack([data[s:s + SEGMENT_LENGTH] for s in starts])
    with torch.inference_mode():
        win_pred = unscale_target(model(torch.tensor(windows, device=device)).cpu().numpy().ravel())

    median = float(np.median(win_pred))
    guess = float(snap(median))
    channel_shift = np.abs(data.mean(axis=0))          # 0 = same average as training, in training std units
    flights.append({
        "name": name,
        "window_time_s": (starts + SEGMENT_LENGTH / 2) / 10.0,   # window centre, 10 Hz rows
        "window_pred": win_pred,
        "median": median,
        "guess": guess,
        "q25": float(np.percentile(win_pred, 25)), "q75": float(np.percentile(win_pred, 75)),
        "agreement": float(np.mean(np.isclose(snap(win_pred), guess))),
        "outside_range": abs(median) > CG_MAX + CG_STEP / 2,
        "data_check": "OK" if channel_shift.max() < 2 else f"UNUSUAL (channel {channel_shift.argmax()})",
        "truth": true_label(name),
        "duration_s": len(data) / 10.0,
    })

def confidence(fl):
    if fl["outside_range"] or fl["data_check"] != "OK":
        return "Low"
    return "High" if fl["agreement"] >= 0.6 else "Medium" if fl["agreement"] >= 0.35 else "Low"

results = pd.DataFrame([{
    "flight": fl["name"],
    "predicted_CG": cg_name(fl["guess"]),
    "predicted_in": round(fl["median"], 3),
    "middle_50%_of_windows_in": f"{fl['q25']:+.2f} to {fl['q75']:+.2f}",
    "agreement": f"{fl['agreement']:.0%}",
    "confidence": confidence(fl),
    "data_check": fl["data_check"],
    "true_CG": cg_name(fl["truth"]) if fl["truth"] is not None else "unknown",
    "correct": (np.isclose(fl["guess"], fl["truth"]) if fl["truth"] is not None else None),
} for fl in flights])

results.to_csv(results_path, index=False)
print(f"Saved {len(results)} predictions to {results_path}\n")
results

### Visual feedback
Colors: blue = the model's guess, orange = the real CG (only when the filename has one).

In [ ]:
BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
CONF_ALPHA = {"High": 1.0, "Medium": 0.6, "Low": 0.3}

plt.rcParams.update({"axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2,
                     "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 10})

def cg_grid(ax, axis="x"):
    """Light lines at each CG position the helicopter can be set to."""
    for c in CG_LEVELS:
        (ax.axvline if axis == "x" else ax.axhline)(c, color=GRID, lw=1, zorder=0)
    ticks = [f"{cg_name(c)}" for c in CG_LEVELS]
    if axis == "x":
        ax.set_xticks(CG_LEVELS); ax.set_xticklabels(ticks, rotation=45)
        ax.set_xlim(-CG_MAX - 0.12, CG_MAX + 0.12)
    else:
        ax.set_yticks(CG_LEVELS); ax.set_yticklabels(ticks)
        ax.set_ylim(-CG_MAX - 0.12, CG_MAX + 0.12)

# ---------- 1) One row per flight: the guess, its window spread, and the truth if known ----------
n = len(flights)
fig, ax = plt.subplots(figsize=(10, max(3, 0.32 * n + 1.2)))
for i, fl in enumerate(flights):
    a = CONF_ALPHA[confidence(fl)]
    ax.plot([fl["q25"], fl["q75"]], [i, i], color=BLUE, lw=2, alpha=a, solid_capstyle="round")
    ax.plot(fl["median"], i, "o", ms=8, color=BLUE, alpha=a, mec="white", mew=2)
    if fl["truth"] is not None:
        ax.plot(fl["truth"], i, "x", ms=9, mew=2, color=ORANGE)
cg_grid(ax, "x")
ax.set_yticks(range(n))
ax.set_yticklabels([f"{fl['name']}  ({confidence(fl)})" for fl in flights], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("CG offset   (aft  <-  ->  forward)")
ax.set_title("Predicted CG per flight  (dot = answer, bar = middle 50% of windows, faded = less confident)", loc="left", color=INK)
handles = [plt.Line2D([], [], color=BLUE, marker="o", lw=2, label="Prediction")]
if any(fl["truth"] is not None for fl in flights):
    handles.append(plt.Line2D([], [], color=ORANGE, marker="x", lw=0, mew=2, label="True CG"))
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, frameon=False)
plt.tight_layout(); plt.show()

### Each flight over time
A steady line means every part of the flight points to the same CG. Big jumps usually line up with takeoff, landing, or hard maneuvers.

In [ ]:
MAX_FLIGHT_PLOTS = 12          # raise this to see more flights

for fl in flights[:MAX_FLIGHT_PLOTS]:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.2), gridspec_kw={"width_ratios": [3, 1]}, sharey=True)
    a1.plot(fl["window_time_s"], fl["window_pred"], color=BLUE, lw=2, marker="o", ms=4)
    a1.axhline(fl["median"], color=BLUE, lw=1, ls="--")
    a1.axhspan(fl["guess"] - CG_STEP / 2, fl["guess"] + CG_STEP / 2, color=BLUE, alpha=0.08, lw=0)
    if fl["truth"] is not None:
        a1.axhline(fl["truth"], color=ORANGE, lw=2)
    cg_grid(a1, "y")
    a1.set_xlabel("Time into flight (s, window centre)")
    a1.set_ylabel("Predicted CG")
    title = f"{fl['name']}:  guess {cg_name(fl['guess'])}  |  agreement {fl['agreement']:.0%}  |  {confidence(fl)} confidence"
    if fl["truth"] is not None:
        title += f"  |  true {cg_name(fl['truth'])}  ({'correct' if np.isclose(fl['guess'], fl['truth']) else 'wrong'})"
    a1.set_title(title, loc="left", color=INK, fontsize=10)

    a2.hist(fl["window_pred"], bins=np.arange(-CG_MAX - CG_STEP, CG_MAX + 2 * CG_STEP, CG_STEP / 2),
            orientation="horizontal", color=BLUE, rwidth=0.85)
    if fl["truth"] is not None:
        a2.axhline(fl["truth"], color=ORANGE, lw=2)
    a2.set_xlabel("Windows")
    a2.set_title("Window votes", loc="left", color=INK, fontsize=10)
    plt.tight_layout(); plt.show()

if len(flights) > MAX_FLIGHT_PLOTS:
    print(f"Showing {MAX_FLIGHT_PLOTS} of {len(flights)} flights - raise MAX_FLIGHT_PLOTS to see the rest")

### Scorecard (only for flights whose filename has the real CG)
To test the model honestly, use flights it was **not** trained on.

In [ ]:
known = [fl for fl in flights if fl["truth"] is not None]
if not known:
    print("None of these filenames contain a CG code, so there's nothing to score. Predictions are above.")
else:
    t = np.array([fl["truth"] for fl in known]); g = np.array([fl["guess"] for fl in known])
    m = np.array([fl["median"] for fl in known])
    acc = np.mean(np.isclose(g, t)); within1 = np.mean(np.abs(g - t) <= CG_STEP + 1e-9)
    mae = np.mean(np.abs(m - t))
    high = [fl for fl in known if confidence(fl) == "High"]
    acc_high = np.mean([np.isclose(fl["guess"], fl["truth"]) for fl in high]) if high else float("nan")

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5.5))

    # predicted vs true
    jit = np.random.default_rng(0).normal(0, 0.012, len(t))
    a1.plot([-CG_MAX, CG_MAX], [-CG_MAX, CG_MAX], color=INK2, lw=1, ls="--", label="Perfect")
    a1.scatter(t + jit, m, s=60, color=BLUE, edgecolor="white", linewidth=2, zorder=3, label="Flight")
    cg_grid(a1, "x"); cg_grid(a1, "y")
    a1.set_xlabel("True CG"); a1.set_ylabel("Predicted CG")
    a1.set_title(f"Predicted vs true  |  MAE {mae:.3f} in", loc="left", color=INK)
    a1.legend(frameon=False, loc="upper left")

    # confusion matrix (counts)
    ti = np.round((t + CG_MAX) / CG_STEP).astype(int); gi = np.round((g + CG_MAX) / CG_STEP).astype(int)
    cm = np.zeros((len(CG_LEVELS), len(CG_LEVELS)), int)
    for a, b in zip(ti, gi):
        cm[a, b] += 1
    a2.imshow(cm, cmap="Blues", origin="lower")
    for d in range(len(CG_LEVELS)):              # outline the correct (diagonal) cells
        a2.add_patch(plt.Rectangle((d - 0.5, d - 0.5), 1, 1, fill=False, ec=ORANGE, lw=1.5))
    for r in range(len(CG_LEVELS)):
        for c in range(len(CG_LEVELS)):
            if cm[r, c]:
                a2.text(c, r, cm[r, c], ha="center", va="center", fontsize=9,
                        color="white" if cm[r, c] > cm.max() / 2 else INK)
    names = [cg_name(c) for c in CG_LEVELS]
    a2.set_xticks(range(len(names))); a2.set_xticklabels(names, rotation=45)
    a2.set_yticks(range(len(names))); a2.set_yticklabels(names)
    a2.set_xlabel("Predicted"); a2.set_ylabel("True")
    a2.set_title("Flights per (true, predicted) pair  -  outlined = correct", loc="left", color=INK)
    a2.spines[:].set_visible(False)
    plt.tight_layout(); plt.show()

    print(f"Flights scored:            {len(known)}")
    print(f"Exact CG position:         {acc:.0%}")
    print(f"Within one position:       {within1:.0%}   (off by at most {CG_STEP:.2f} in)")
    print(f"High-confidence only:      " + (f"{acc_high:.0%}   ({len(high)} flights)" if high else "no high-confidence flights"))
    print(f"Mean error:                {mae:.3f} in")